In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))

class WeightedL1(nn.Module):
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()

def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

print("✅ Components ready")

✅ Components ready


In [3]:
# Load data
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98,
    batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

test_loader = load_test_data(
    test_path="../datasets_challenge/test",
    batchsize=512,
    pipeline=pipeline,
    load_anonymized_data=True
)

X_sample, _ = next(iter(train_loader))
input_size = X_sample.shape[2]
print(f"✅ Data loaded | Input size: {input_size}")

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.csv
../datasets_challenge/train/train_part_015.csv
../datasets_challenge/train/train_part_016.c

In [4]:
def train_two_stage(seed, s2_epochs):
    """S1: L1@3e-4, 30ep | S2: WeightedL1@5e-5, variable epochs"""
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1 (FIXED)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2 (variable epochs)
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    print(f"    seed={seed} S2:{s2_epochs}ep → val={best2:.4f}")
    return model

def train_baseline(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    print(f"    seed={seed} baseline → val={best:.4f}")
    return model

print("✅ Functions ready")

✅ Functions ready


## 🚀 Train All Models

In [5]:
# All seeds we'll need
ALL_SEEDS = [42, 2024, 123, 456]

print("="*60)
print("📦 TRAINING ALL BASELINE MODELS (4 seeds)")
print("="*60)

baseline_preds = {}
for seed in ALL_SEEDS:
    model = train_baseline(seed)
    baseline_preds[seed] = predict(model, test_loader)

print("✅ All baselines done")

📦 TRAINING ALL BASELINE MODELS (4 seeds)
    seed=42 baseline → val=0.1137
    seed=42 baseline → val=0.1137
    seed=2024 baseline → val=0.1116
    seed=2024 baseline → val=0.1116
    seed=123 baseline → val=0.1190
    seed=123 baseline → val=0.1190
    seed=456 baseline → val=0.1263
    seed=456 baseline → val=0.1263
✅ All baselines done
✅ All baselines done


In [6]:
print("\n" + "="*60)
print("📦 TRAINING TWO-STAGE MODELS")
print("="*60)

twostage_preds = {}

# 38ep and 42ep (epoch experiments) - only need 2 seeds
for epochs in [38, 42]:
    print(f"\n📦 {epochs} epochs:")
    twostage_preds[epochs] = {}
    for seed in [42, 2024]:
        model = train_two_stage(seed, epochs)
        twostage_preds[epochs][seed] = predict(model, test_loader)

# 40ep for all 4 seeds (for blend ratios + multi-seed experiments)
print(f"\n📦 40 epochs (all 4 seeds):")
twostage_preds[40] = {}
for seed in ALL_SEEDS:
    model = train_two_stage(seed, 40)
    twostage_preds[40][seed] = predict(model, test_loader)

print("\n✅ All two-stage models done")


📦 TRAINING TWO-STAGE MODELS

📦 38 epochs:
    seed=42 S2:38ep → val=0.1141
    seed=42 S2:38ep → val=0.1141
    seed=2024 S2:38ep → val=0.1102
    seed=2024 S2:38ep → val=0.1102

📦 42 epochs:

📦 42 epochs:
    seed=42 S2:42ep → val=0.1127
    seed=42 S2:42ep → val=0.1127
    seed=2024 S2:42ep → val=0.1119
    seed=2024 S2:42ep → val=0.1119

📦 40 epochs (all 4 seeds):

📦 40 epochs (all 4 seeds):
    seed=42 S2:40ep → val=0.1143
    seed=42 S2:40ep → val=0.1143
    seed=2024 S2:40ep → val=0.1123
    seed=2024 S2:40ep → val=0.1123
    seed=123 S2:40ep → val=0.1137
    seed=123 S2:40ep → val=0.1137
    seed=456 S2:40ep → val=0.1130
    seed=456 S2:40ep → val=0.1130

✅ All two-stage models done

✅ All two-stage models done


## 💾 Generate 6 Prediction Files

In [7]:
print("\n" + "="*60)
print("💾 GENERATING 6 PREDICTION FILES")
print("="*60)

# Helper: create ensemble from seeds
def make_ensemble(preds_dict, seeds):
    return np.mean([preds_dict[s] for s in seeds], axis=0)

# 2-seed ensembles for experiments 1-4
baseline_2seed = make_ensemble(baseline_preds, [42, 2024])
ts_40ep_2seed = make_ensemble(twostage_preds[40], [42, 2024])

# ============================================================
# FILE 1: 38 epochs (50/50 blend, 2 seeds)
# ============================================================
ts_38ep = make_ensemble(twostage_preds[38], [42, 2024])
blend_38ep = (baseline_2seed + ts_38ep) / 2
pd.DataFrame({"id": range(len(blend_38ep)), "predicted_sideslip": blend_38ep}).to_csv(
    "pred_v9_38ep.csv", index=False)
print("✅ 1/6: pred_v9_38ep.csv")

# ============================================================
# FILE 2: 42 epochs (50/50 blend, 2 seeds)
# ============================================================
ts_42ep = make_ensemble(twostage_preds[42], [42, 2024])
blend_42ep = (baseline_2seed + ts_42ep) / 2
pd.DataFrame({"id": range(len(blend_42ep)), "predicted_sideslip": blend_42ep}).to_csv(
    "pred_v9_42ep.csv", index=False)
print("✅ 2/6: pred_v9_42ep.csv")

# ============================================================
# FILE 3: 45/55 blend (more two-stage, 40ep, 2 seeds)
# ============================================================
blend_45_55 = baseline_2seed * 0.45 + ts_40ep_2seed * 0.55
pd.DataFrame({"id": range(len(blend_45_55)), "predicted_sideslip": blend_45_55}).to_csv(
    "pred_v9_40ep_b45_ts55.csv", index=False)
print("✅ 3/6: pred_v9_40ep_b45_ts55.csv")

# ============================================================
# FILE 4: 55/45 blend (more baseline, 40ep, 2 seeds)
# ============================================================
blend_55_45 = baseline_2seed * 0.55 + ts_40ep_2seed * 0.45
pd.DataFrame({"id": range(len(blend_55_45)), "predicted_sideslip": blend_55_45}).to_csv(
    "pred_v9_40ep_b55_ts45.csv", index=False)
print("✅ 4/6: pred_v9_40ep_b55_ts45.csv")

# ============================================================
# FILE 5: 3 seeds (40ep, 50/50 blend)
# ============================================================
baseline_3seed = make_ensemble(baseline_preds, [42, 2024, 123])
ts_40ep_3seed = make_ensemble(twostage_preds[40], [42, 2024, 123])
blend_3seed = (baseline_3seed + ts_40ep_3seed) / 2
pd.DataFrame({"id": range(len(blend_3seed)), "predicted_sideslip": blend_3seed}).to_csv(
    "pred_v9_40ep_3seeds.csv", index=False)
print("✅ 5/6: pred_v9_40ep_3seeds.csv")

# ============================================================
# FILE 6: 4 seeds (40ep, 50/50 blend)
# ============================================================
baseline_4seed = make_ensemble(baseline_preds, ALL_SEEDS)
ts_40ep_4seed = make_ensemble(twostage_preds[40], ALL_SEEDS)
blend_4seed = (baseline_4seed + ts_40ep_4seed) / 2
pd.DataFrame({"id": range(len(blend_4seed)), "predicted_sideslip": blend_4seed}).to_csv(
    "pred_v9_40ep_4seeds.csv", index=False)
print("✅ 6/6: pred_v9_40ep_4seeds.csv")


💾 GENERATING 6 PREDICTION FILES
✅ 1/6: pred_v9_38ep.csv
✅ 2/6: pred_v9_42ep.csv
✅ 3/6: pred_v9_40ep_b45_ts55.csv
✅ 4/6: pred_v9_40ep_b55_ts45.csv
✅ 5/6: pred_v9_40ep_3seeds.csv
✅ 6/6: pred_v9_40ep_4seeds.csv
✅ 4/6: pred_v9_40ep_b55_ts45.csv
✅ 5/6: pred_v9_40ep_3seeds.csv
✅ 6/6: pred_v9_40ep_4seeds.csv


In [8]:
# Prediction statistics
print("\n" + "="*60)
print("📊 PREDICTION STATISTICS")
print("="*60)

def stats(name, p):
    print(f"{name}: mean={p.mean():.3f}, std={p.std():.3f}, range=[{p.min():.2f}, {p.max():.2f}]")

stats("38ep      ", blend_38ep)
stats("42ep      ", blend_42ep)
stats("45/55     ", blend_45_55)
stats("55/45     ", blend_55_45)
stats("3 seeds   ", blend_3seed)
stats("4 seeds   ", blend_4seed)


📊 PREDICTION STATISTICS
38ep      : mean=-0.003, std=1.494, range=[-9.53, 9.66]
42ep      : mean=0.002, std=1.493, range=[-9.41, 9.51]
45/55     : mean=-0.000, std=1.491, range=[-9.42, 9.54]
55/45     : mean=-0.002, std=1.495, range=[-9.43, 9.52]
3 seeds   : mean=0.007, std=1.488, range=[-9.56, 9.61]
4 seeds   : mean=0.011, std=1.481, range=[-9.60, 9.67]


In [9]:
print("\n" + "="*60)
print("📋 SUMMARY: 6 FILES TO SUBMIT")
print("="*60)

print("\n🏆 Current Best: pred_v7_lr5e5_40ep → 0.21751")

print("\n📁 Files:")
print("  1. pred_v9_38ep.csv         (38 epochs, 2 seeds, 50/50)")
print("  2. pred_v9_42ep.csv         (42 epochs, 2 seeds, 50/50)")
print("  3. pred_v9_40ep_b45_ts55.csv (40ep, more two-stage)")
print("  4. pred_v9_40ep_b55_ts45.csv (40ep, more baseline)")
print("  5. pred_v9_40ep_3seeds.csv  (40ep, 3 seeds)")
print("  6. pred_v9_40ep_4seeds.csv  (40ep, 4 seeds)")

print("\n💡 Priority:")
print("  🥇 pred_v9_38ep.csv - might hit sweet spot before overfitting")
print("  🥈 pred_v9_40ep_b45_ts55.csv - slight boost to two-stage")
print("  🥉 pred_v9_40ep_3seeds.csv - more diversity")


📋 SUMMARY: 6 FILES TO SUBMIT

🏆 Current Best: pred_v7_lr5e5_40ep → 0.21751

📁 Files:
  1. pred_v9_38ep.csv         (38 epochs, 2 seeds, 50/50)
  2. pred_v9_42ep.csv         (42 epochs, 2 seeds, 50/50)
  3. pred_v9_40ep_b45_ts55.csv (40ep, more two-stage)
  4. pred_v9_40ep_b55_ts45.csv (40ep, more baseline)
  5. pred_v9_40ep_3seeds.csv  (40ep, 3 seeds)
  6. pred_v9_40ep_4seeds.csv  (40ep, 4 seeds)

💡 Priority:
  🥇 pred_v9_38ep.csv - might hit sweet spot before overfitting
  🥈 pred_v9_40ep_b45_ts55.csv - slight boost to two-stage
  🥉 pred_v9_40ep_3seeds.csv - more diversity
